# 🚀 ASTRA / TejaAI — Free GPU Model Server (Google Colab)

This notebook runs your fine-tuned **Gemma 4 E2B + TejaAI LoRA Adapter** on a free Colab **Tesla T4 GPU** and creates a secure public tunnel so your local laptop ASTRA agent (`astra.py`) can use it!

### Instructions:
1. **Runtime** → **Change runtime type** → select **T4 GPU**.
2. Run all cells from top to bottom.
3. Copy the generated **Cloudflare Public URL** into your local laptop's `astra-2.0/.env` file.

In [ ]:
# 2. Install required packages (Install Transformers with Gemma 4 support directly from GitHub)
!pip install -q --upgrade "git+https://github.com/huggingface/transformers.git" "peft>=0.14.0" "accelerate>=1.2.0" fastapi uvicorn pydantic


In [ ]:
# 3. Hugging Face Cloud Adapter (No Google Drive or file upload needed!)
ADAPTER_PATH = "Tulasiteja9/teja-ai-gemma4-e2b-lora"
HF_TOKEN = os.getenv("HF_TOKEN", "") or "YOUR_HF_TOKEN_HERE"
print(f"✅ Adapter Repo: https://huggingface.co/{ADAPTER_PATH}")


In [ ]:
# 4. Load Base Model + LoRA Adapter
import torch
from transformers import AutoTokenizer
from peft import PeftModel

BASE_MODEL_ID = "google/gemma-4-E2B"

print('Loading Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading Base Gemma Model on T4 GPU...')
try:
    from transformers import AutoModelForImageTextToText
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map={'': 0} if torch.cuda.is_available() else None,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
except Exception:
    from transformers import AutoModelForCausalLM
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map={'': 0} if torch.cuda.is_available() else None,
        trust_remote_code=True,
        token=HF_TOKEN,
    )

print('Attaching Fine-Tuned LoRA Adapter from Hugging Face...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, token=HF_TOKEN)
model.eval()
print('✅ TejaAI Gemma Fine-Tuned Model successfully loaded on GPU!')


In [ ]:
# 4. Load Base Model + LoRA Adapter in 4-bit NF4
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "google/gemma-4-E2B"
TOKENIZER_ID = "google/gemma-4-E2B-it"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading Base Gemma Model on T4 GPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Attaching Fine-Tuned LoRA Adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("✅ TejaAI Gemma Fine-Tuned Model successfully loaded on GPU!")

In [ ]:
# 5. Create OpenAI-Compatible FastAPI Server
import time
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title="TejaAI Colab Model Server")

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = "teja-ai-gemma4-e2b"
    messages: List[Message]
    temperature: Optional[float] = 0.2
    max_tokens: Optional[int] = 512

@app.get("/health")
def health():
    return {"status": "ok", "model": "teja-ai-gemma4-e2b"}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    chat_msgs = [{"role": m.role, "content": m.content} for m in req.messages]
    prompt = tokenizer.apply_chat_template(chat_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    turn_id = tokenizer.convert_tokens_to_ids("<turn|>")
    eos_ids = [tokenizer.eos_token_id]
    if turn_id and turn_id != tokenizer.unk_token_id:
        eos_ids.append(turn_id)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens or 128,
            do_sample=req.temperature > 0.05,
            temperature=req.temperature,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": len(inputs["input_ids"][0]),
            "completion_tokens": len(new_tokens),
            "total_tokens": len(inputs["input_ids"][0]) + len(new_tokens),
        },
    }


In [ ]:
# 6. Clean port 8000 & Launch High-Speed Uvicorn + Cloudflare Tunnel
!fuser -k 8000/tcp 2>/dev/null || true
!pkill -9 -f uvicorn 2>/dev/null || true
!pkill -9 -f cloudflared 2>/dev/null || true
import os, time, subprocess, threading, traceback
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Union
from transformers import StoppingCriteria, StoppingCriteriaList
import uvicorn
time.sleep(2)

# Collect all possible stop token IDs for Gemma
stop_token_ids = set()
if tokenizer.eos_token_id is not None:
    stop_token_ids.add(tokenizer.eos_token_id)
if tokenizer.pad_token_id is not None:
    stop_token_ids.add(tokenizer.pad_token_id)
for st in ['<end_of_turn>', '<turn|>', '<|turn>', '<tool_call|>', '<eos>', '<|endoftext|>']:
    tid = tokenizer.convert_tokens_to_ids(st)
    if tid is not None and tid != getattr(tokenizer, 'unk_token_id', None):
        stop_token_ids.add(tid)
print(f'Configured Stop Token IDs: {stop_token_ids}')

class FastAgentStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len, stop_ids):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.stop_ids = stop_ids
        self.stop_phrases = ['</tool_call>', '<turn|>', '<end_of_turn>', '<tool_call|>', '```\n\nIn [', '\n\nUser:']
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        last_id = input_ids[0, -1].item()
        if last_id in self.stop_ids:
            return True
        curr_len = input_ids.shape[1]
        new_tokens_count = curr_len - self.prompt_len
        if new_tokens_count >= 4 and new_tokens_count % 2 == 0:
            tail_ids = input_ids[0, max(self.prompt_len, curr_len - 12):]
            tail_str = self.tokenizer.decode(tail_ids, skip_special_tokens=False)
            for phrase in self.stop_phrases:
                if phrase in tail_str:
                    return True
        return False

app = FastAPI(title='TejaAI Colab Model Server')

class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = 'teja-ai-gemma4-e2b'
    messages: List[Message]
    temperature: Optional[float] = 0.1
    max_tokens: Optional[int] = 160
    stop: Optional[Union[str, List[str]]] = None

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'teja-ai-gemma4-e2b'}

@app.post('/v1/chat/completions')
def chat_completions(req: ChatCompletionRequest):
    try:
        chat_msgs = []
        system_text = ''
        for m in req.messages:
            if m.role == 'system':
                system_text += m.content + '\n\n'
            else:
                chat_msgs.append({'role': m.role, 'content': m.content})
        if system_text:
            if chat_msgs and chat_msgs[0]['role'] == 'user':
                chat_msgs[0]['content'] = f"{system_text.strip()}\n\n{chat_msgs[0]['content']}"
            else:
                chat_msgs.insert(0, {'role': 'user', 'content': system_text.strip()})
        if not chat_msgs:
            chat_msgs = [{'role': 'user', 'content': 'Hello'}]
        prompt = tokenizer.apply_chat_template(chat_msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')
        prompt_len = inputs['input_ids'].shape[1]
        stopping_criteria = StoppingCriteriaList([FastAgentStoppingCriteria(tokenizer, prompt_len, stop_token_ids)])
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens or 160,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=stopping_criteria,
            )
        new_tokens = out[0][prompt_len:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=False).strip()
        for stop_word in ['<turn|>', '<|turn>', '<tool_call|>', '<end_of_turn>', '<eos>', '<|endoftext|>']:
            if stop_word in text:
                text = text.split(stop_word)[0].strip()
        if '</tool_call>' in text:
            text = text.split('</tool_call>')[0] + '</tool_call>'
        if '```\n\nIn [' in text:
            text = text.split('```\n\nIn [')[0].strip()
        return {
            'id': f'chatcmpl-{int(time.time())}',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': req.model,
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': text},
                'finish_reason': 'stop',
            }],
            'usage': {
                'prompt_tokens': prompt_len,
                'completion_tokens': len(new_tokens),
                'total_tokens': prompt_len + len(new_tokens),
            },
        }
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f'Error: {e}')

def start_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(3)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

print('\nStarting Cloudflare Tunnel...')
tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
public_url = None
for _ in range(30):
    line = tunnel_process.stderr.readline()
    if 'trycloudflare.com' in line:
        import re
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)
if public_url:
    print('\n' + '='*60)
    print('🎉 YOUR HIGH-SPEED MODEL SERVER IS LIVE!')
    print(f'Public URL: {public_url}/v1')
    print('='*60)
    print(f'\nTEJA_MODEL_URL={public_url}/v1\n')
    print('Keep-alive active. Server running in background...')
    while True:
        time.sleep(60)
else:
    print('❌ Failed to obtain tunnel URL. Check cloudflared output.')
